# JS09 Lab 3: Exact NN vs HNSW

Jobsheet 09, Lab 3. Percobaan ini mengulang perbandingan yang sama, tetapi algoritma ANN yang dipakai adalah HNSW dari `hnswlib`. HNSW menyusun data sebagai graf bertingkat: lapisan atas berisi sedikit titik untuk lompatan jauh, lapisan bawah berisi semua titik untuk pencarian halus.

## Langkah 1 - Instal Library

In [ ]:
!pip install hnswlib

## Langkah 2 - Import Library

In [1]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

## Langkah 3 - Buat Dataset Dummy

Kali ini `k=5`, jadi exact NN dan HNSW sama-sama mencari lima tetangga terdekat.

In [2]:
# Build Dummy Dataset
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # cari 5 tetangga terdekat

## Langkah 4 - Buat Model NN

`algorithm='brute'` memaksa scikit-learn membandingkan query dengan seluruh data, sehingga hasilnya menjadi acuan exact.

In [3]:
# Build NN Model
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

# Compute time performace
start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "s")

=== Exact NN ===
Indices: [[647  27 642 259 684]]
Distances: [[0.0106705  0.01481052 0.02200506 0.04845904 0.04956017]]
Waktu: 2.2097089290618896 s


## Langkah 5 - Buat Model HNSW

`ef_construction` mengatur kualitas graf saat pembangunan, sedangkan `ef` mengatur seberapa banyak kandidat yang diperiksa saat query. Nilai `ef` yang lebih besar menaikkan akurasi dengan konsekuensi waktu pencarian bertambah.

In [4]:
# Initiate Index
p = hnswlib.Index(space='l2', dim=dim)  

# Define Max Elements
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Add Data
p.add_items(data)

# Set searching parameter
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "s")


=== HNSW ===
Indices: [[647  27 642 259 684]]
Distances: [[0.00011386 0.00021935 0.00048422 0.00234828 0.00245621]]
Waktu: 0.0 s


Indeks yang ditemukan HNSW bisa sama persis dengan exact NN, tetapi jarak yang dilaporkan HNSW adalah kuadrat jarak L2. Waktu query HNSW jauh lebih kecil karena graf hanya dikunjungi pada sebagian kecil titik.